In [0]:
from collections import defaultdict
from pyspark.sql.functions import to_date
import re

files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/customer/")

grouped = defaultdict(list)

for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    
    latest = sorted_files[0]
    old_files = sorted_files[1:]
    
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        # Table doesn't exist - create it from the latest file
        print(f"Table {table_name} does not exist. Creating new table...")
        
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(latest)
        
        # Cast date columns if they exist
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        df_new.write.mode("append").saveAsTable(table_name)
        print(f"Created new table {table_name} with {df_new.count()} records from {latest.split('/')[-1]}")
    
    else:
        # Table exists - check if it's Delta or external, then append
        print(f"Table {table_name} exists. Processing new data...")
        
        # Load the latest file data
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(latest)
        
        # Cast date columns if they exist
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        # Check if it's a Delta table
        try:
            table_type = spark.sql(f"DESCRIBE DETAIL {table_name}").select("format").collect()[0][0]
            is_delta = table_type.lower() == "delta"
        except:
            # If DESCRIBE DETAIL fails, it's not a Delta table
            is_delta = False
        
        if is_delta:
            # It's already a Delta table, just append
            df_new.write.mode("append").saveAsTable(table_name)
            print(f"Appended {df_new.count()} records from {latest.split('/')[-1]} to {table_name}")
        else:
            # It's an external table, need to convert to Delta while preserving data
            print(f"Converting {table_name} from external to Delta table...")
            
            # Read existing data and cast date columns if they exist
            df_existing = spark.table(table_name)
            if "LastUpdated" in df_existing.columns:
                df_existing = df_existing.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
            if "TxnDate" in df_existing.columns:
                df_existing = df_existing.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
            
            temp_table = f"{table_name}_temp"
            df_existing.write.mode("overwrite").saveAsTable(temp_table)
            existing_count = spark.table(temp_table).count()
            
            # Combine existing and new data
            df_combined = spark.table(temp_table).union(df_new)
            
            # Drop the old external table
            spark.sql(f"DROP TABLE {table_name}")
            
            # Create new Delta table with all data
            df_combined.write.mode("overwrite").saveAsTable(table_name)
            print(f"Converted table and loaded {df_combined.count()} total records (existing: {existing_count}, new: {df_new.count()})")
            
            # Clean up temp table
            spark.sql(f"DROP TABLE {temp_table}")
    
    # Archive old files
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        
        dbutils.fs.mv(
            old_file,
            f"s3://retail-etl-project-revanth/archive/customer/{file_name}"
        )
    
    print(f"Archived {len(old_files)} old file(s) for {dataset}")

In [0]:
%sql
SELECT count(*)
FROM retail_catalog.bronze.customers_raw

In [0]:
from collections import defaultdict
from pyspark.sql.functions import to_date
import re

files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/products/")

grouped = defaultdict(list)

for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    
    latest = sorted_files[0]
    old_files = sorted_files[1:]
    
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        # Table doesn't exist - create it from the latest file
        print(f"Table {table_name} does not exist. Creating new table...")
        
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(latest)
        
        # Cast date columns if they exist
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        df_new.write.mode("append").saveAsTable(table_name)
        print(f"Created new table {table_name} with {df_new.count()} records from {latest.split('/')[-1]}")
    
    else:
        # Table exists - check if it's Delta or external, then append
        print(f"Table {table_name} exists. Processing new data...")
        
        # Load the latest file data
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(latest)
        
        # Cast date columns if they exist
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        # Check if it's a Delta table
        try:
            table_type = spark.sql(f"DESCRIBE DETAIL {table_name}").select("format").collect()[0][0]
            is_delta = table_type.lower() == "delta"
        except:
            # If DESCRIBE DETAIL fails, it's not a Delta table
            is_delta = False
        
        if is_delta:
            # It's already a Delta table, just append
            df_new.write.mode("append").saveAsTable(table_name)
            print(f"Appended {df_new.count()} records from {latest.split('/')[-1]} to {table_name}")
        else:
            # It's an external table, need to convert to Delta while preserving data
            print(f"Converting {table_name} from external to Delta table...")
            
            # Read existing data and cast date columns if they exist
            df_existing = spark.table(table_name)
            if "LastUpdated" in df_existing.columns:
                df_existing = df_existing.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
            if "TxnDate" in df_existing.columns:
                df_existing = df_existing.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
            
            temp_table = f"{table_name}_temp"
            df_existing.write.mode("overwrite").saveAsTable(temp_table)
            existing_count = spark.table(temp_table).count()
            
            # Combine existing and new data
            df_combined = spark.table(temp_table).union(df_new)
            
            # Drop the old external table
            spark.sql(f"DROP TABLE {table_name}")
            
            # Create new Delta table with all data
            df_combined.write.mode("overwrite").saveAsTable(table_name)
            print(f"Converted table and loaded {df_combined.count()} total records (existing: {existing_count}, new: {df_new.count()})")
            
            # Clean up temp table
            spark.sql(f"DROP TABLE {temp_table}")
    
    # Archive old files
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        
        dbutils.fs.mv(
            old_file,
            f"s3://retail-etl-project-revanth/archive/products/{file_name}"
        )
    
    print(f"Archived {len(old_files)} old file(s) for {dataset}")

In [0]:
from collections import defaultdict
from pyspark.sql.functions import to_date
import re

files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/sales/")

grouped = defaultdict(list)

for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    
    latest = sorted_files[0]
    old_files = sorted_files[1:]
    
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        # Table doesn't exist - create it from the latest file
        print(f"Table {table_name} does not exist. Creating new table...")
        
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(latest)
        
        # Cast date columns if they exist
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        df_new.write.mode("append").saveAsTable(table_name)
        print(f"Created new table {table_name} with {df_new.count()} records from {latest.split('/')[-1]}")
    
    else:
        # Table exists - check if it's Delta or external, then append
        print(f"Table {table_name} exists. Processing new data...")
        
        # Load the latest file data
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(latest)
        
        # Cast date columns if they exist
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        # Check if it's a Delta table
        try:
            table_type = spark.sql(f"DESCRIBE DETAIL {table_name}").select("format").collect()[0][0]
            is_delta = table_type.lower() == "delta"
        except:
            # If DESCRIBE DETAIL fails, it's not a Delta table
            is_delta = False
        
        if is_delta:
            # It's already a Delta table, just append
            df_new.write.mode("append").saveAsTable(table_name)
            print(f"Appended {df_new.count()} records from {latest.split('/')[-1]} to {table_name}")
        else:
            # It's an external table, need to convert to Delta while preserving data
            print(f"Converting {table_name} from external to Delta table...")
            
            # Read existing data and cast date columns if they exist
            df_existing = spark.table(table_name)
            if "LastUpdated" in df_existing.columns:
                df_existing = df_existing.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
            if "TxnDate" in df_existing.columns:
                df_existing = df_existing.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
            
            temp_table = f"{table_name}_temp"
            df_existing.write.mode("overwrite").saveAsTable(temp_table)
            existing_count = spark.table(temp_table).count()
            
            # Combine existing and new data
            df_combined = spark.table(temp_table).union(df_new)
            
            # Drop the old external table
            spark.sql(f"DROP TABLE {table_name}")
            
            # Create new Delta table with all data
            df_combined.write.mode("overwrite").saveAsTable(table_name)
            print(f"Converted table and loaded {df_combined.count()} total records (existing: {existing_count}, new: {df_new.count()})")
            
            # Clean up temp table
            spark.sql(f"DROP TABLE {temp_table}")
    
    # Archive old files
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        
        dbutils.fs.mv(
            old_file,
            f"s3://retail-etl-project-revanth/archive/sales/{file_name}"
        )
    
    print(f"Archived {len(old_files)} old file(s) for {dataset}")

In [0]:
from collections import defaultdict
from pyspark.sql.functions import to_date
import re

files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/stores/")

grouped = defaultdict(list)

for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    
    latest = sorted_files[0]
    old_files = sorted_files[1:]
    
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    # Check if table exists
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        # Table doesn't exist - create it from the latest file
        print(f"Table {table_name} does not exist. Creating new table...")
        
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(latest)
        
        # Cast date columns if they exist
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        df_new.write.mode("append").saveAsTable(table_name)
        print(f"Created new table {table_name} with {df_new.count()} records from {latest.split('/')[-1]}")
    
    else:
        # Table exists - check if it's Delta or external, then append
        print(f"Table {table_name} exists. Processing new data...")
        
        # Load the latest file data
        df_new = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(latest)
        
        # Cast date columns if they exist
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        # Check if it's a Delta table
        try:
            table_type = spark.sql(f"DESCRIBE DETAIL {table_name}").select("format").collect()[0][0]
            is_delta = table_type.lower() == "delta"
        except:
            # If DESCRIBE DETAIL fails, it's not a Delta table
            is_delta = False
        
        if is_delta:
            # It's already a Delta table, just append
            df_new.write.mode("append").saveAsTable(table_name)
            print(f"Appended {df_new.count()} records from {latest.split('/')[-1]} to {table_name}")
        else:
            # It's an external table, need to convert to Delta while preserving data
            print(f"Converting {table_name} from external to Delta table...")
            
            # Read existing data and cast date columns if they exist
            df_existing = spark.table(table_name)
            if "LastUpdated" in df_existing.columns:
                df_existing = df_existing.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
            if "TxnDate" in df_existing.columns:
                df_existing = df_existing.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
            
            temp_table = f"{table_name}_temp"
            df_existing.write.mode("overwrite").saveAsTable(temp_table)
            existing_count = spark.table(temp_table).count()
            
            # Combine existing and new data
            df_combined = spark.table(temp_table).union(df_new)
            
            # Drop the old external table
            spark.sql(f"DROP TABLE {table_name}")
            
            # Create new Delta table with all data
            df_combined.write.mode("overwrite").saveAsTable(table_name)
            print(f"Converted table and loaded {df_combined.count()} total records (existing: {existing_count}, new: {df_new.count()})")
            
            # Clean up temp table
            spark.sql(f"DROP TABLE {temp_table}")
    
    # Archive old files
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        
        dbutils.fs.mv(
            old_file,
            f"s3://retail-etl-project-revanth/archive/stores/{file_name}"
        )
    
    print(f"Archived {len(old_files)} old file(s) for {dataset}")